# Training pipeline

### Setup

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
print(ROOT)

In [ ]:
from datetime import date, timedelta
import gc
import json
import os
import shutil
import subprocess
import time

from dotenv import load_dotenv
import mlflow
import numba
import numpy as np
import polars as pl
from scipy.optimize import nnls
from scipy.special import ndtri
import torch
from torch import nn

DATA = ROOT / "data"
MODELS = ROOT / "models"
ARCHIVE = ROOT / "archive"
SETUP = DATA / "setup"
FEAT = DATA / "features"
SEQ = DATA / "seq"
MEMBER_DIR = MODELS / "members"
COMBINER_DIR = MODELS / "combiner"
for d in (
    DATA,
    MODELS,
    SETUP,
    FEAT,
    FEAT / "x",
    FEAT / "e",
    FEAT / "f",
    FEAT / "rg",
    SEQ,
    MEMBER_DIR,
    COMBINER_DIR,
):
    d.mkdir(parents=True, exist_ok=True)

VPS_IP = "2.26.27.187"
EXPERIMENT = "ecup-prod"
load_dotenv(ROOT / ".env")
mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", f"https://{VPS_IP}"))
mlflow.set_experiment(EXPERIMENT)
mlflow.enable_system_metrics_logging()
mlflow.set_system_metrics_sampling_interval(5)
mlflow.set_system_metrics_samples_before_logging(2)
print(mlflow.get_tracking_uri(), EXPERIMENT, torch.cuda.is_available())

### Download

In [ ]:
TRAIN_PATH = DATA / "train.parquet"
if not TRAIN_PATH.exists():
    curl = shutil.which("curl")
    assert curl, "curl not found"
    part = TRAIN_PATH.with_suffix(".parquet.part")
    subprocess.run(  # noqa: S603
        [
            curl,
            "-sk",
            "-L",
            "--retry",
            "30",
            "--retry-all-errors",
            "--retry-delay",
            "2",
            "--create-dirs",
            "-o",
            str(part),
            f"https://{VPS_IP}/files/total.parquet",
        ],
        check=True,
    )
    assert part.stat().st_size > 0
    pl.read_parquet(part, n_rows=1)
    part.replace(TRAIN_PATH)
print(TRAIN_PATH, TRAIN_PATH.stat().st_size, "bytes")

### Load

In [ ]:
data = pl.read_parquet(TRAIN_PATH).sort("user_id", "event_date")
DATE_MIN, DATE_MAX = data["event_date"].min(), data["event_date"].max()
print(data.shape, DATE_MIN, "to", DATE_MAX, data["user_id"].n_unique(), "users")

### Anchors, cohorts and targets

In [ ]:
HORIZON_DAYS = 30
MIN_HISTORY_DAYS = 60
CLEAN = ["2025-09-24", "2025-10-08", "2025-10-22"]
RECENT = ["2025-12-03", "2025-12-17", "2025-12-31", "2026-01-14"]
VAL_ANCHORS = [*CLEAN, *RECENT]
PANEL = ["2025-08-27", "2025-10-08", "2025-11-12", "2025-12-17"]
EARLY_ANCHORS = [date(2025, 3, 12), date(2025, 4, 9), date(2025, 5, 7), date(2025, 6, 4)]

TRAIN_DATES = list(EARLY_ANCHORS)
TRAIN_DATES += [date(2025, 7, 2) + timedelta(days=14 * k) for k in range(7)]
TRAIN_DATES += [date(2025, 10, 1) + timedelta(days=7 * k) for k in range(16)]
TRAIN_DATES = sorted(TRAIN_DATES)
assert sorted(set(TRAIN_DATES)) == TRAIN_DATES
assert all(a + timedelta(days=HORIZON_DAYS) <= DATE_MAX for a in TRAIN_DATES)
ALL_DATES = [*TRAIN_DATES, DATE_MAX]
ANCHORS = [a.isoformat() for a in TRAIN_DATES]
SUBMIT = DATE_MAX.isoformat()
ANCHOR_IDX = {a: i for i, a in enumerate(ANCHORS)}
print(len(ANCHORS), "train anchors,", ANCHORS[0], "to", ANCHORS[-1], "submit", SUBMIT)

In [ ]:
CONFIG_PATH = SETUP / "config.json"
if not CONFIG_PATH.exists():
    users = data["user_id"].unique().sort().to_numpy()
    n_users = len(users)
    rows = np.searchsorted(users, data["user_id"].to_numpy())
    day = (
        data
        .select((pl.col("event_date") - pl.lit(DATE_MIN)).dt.total_days().cast(pl.Int32))
        .to_series()
        .to_numpy()
    )
    gmv_all = data["gmv"].to_numpy().astype(np.float64)
    first_day = day[np.searchsorted(rows, np.arange(n_users))]

    cohort = np.zeros((len(ALL_DATES), n_users), dtype=bool)
    targets = np.zeros((len(TRAIN_DATES), n_users), dtype=np.float64)
    active_next = np.zeros((len(TRAIN_DATES), n_users), dtype=bool)
    for i, anchor in enumerate(ALL_DATES):
        t = (anchor - DATE_MIN).days
        recent = (day > t - HORIZON_DAYS) & (day <= t)
        cohort[i] = (np.bincount(rows[recent], minlength=n_users) > 0) & (
            first_day <= t - MIN_HISTORY_DAYS
        )
        if i < len(TRAIN_DATES):
            nxt = (day > t) & (day <= t + HORIZON_DAYS)
            targets[i] = np.bincount(rows[nxt], weights=gmv_all[nxt], minlength=n_users)
            active_next[i] = np.bincount(rows[nxt], minlength=n_users) > 0

    levels = []
    for i, anchor in enumerate(TRAIN_DATES):
        m = cohort[i]
        y = targets[i][m]
        ly = np.log1p(y)
        levels.append({
            "anchor": anchor.isoformat(),
            "n_cohort": int(m.sum()),
            "p_active_next": float(active_next[i][m].mean()),
            "p_positive": float((y > 0).mean()),
            "e_log_given_positive": float(ly[y > 0].mean()),
            "mean_log1p": float(ly.mean()),
        })

    np.save(SETUP / "cohort.npy", cohort)
    np.save(SETUP / "targets.npy", targets)
    np.save(SETUP / "active_next.npy", active_next)
    np.save(SETUP / "user_ids.npy", users)
    (SETUP / "levels.json").write_text(json.dumps(levels, indent=1))
    CONFIG_PATH.write_text(
        json.dumps(
            {
                "date_min": DATE_MIN.isoformat(),
                "date_max": DATE_MAX.isoformat(),
                "horizon_days": HORIZON_DAYS,
                "min_history_days": MIN_HISTORY_DAYS,
                "n_users": n_users,
                "train_anchors": ANCHORS,
                "submit_anchor": SUBMIT,
            },
            indent=2,
        )
    )
    del cohort, targets, active_next, rows, day, gmv_all, first_day
    gc.collect()

CONFIG = json.loads(CONFIG_PATH.read_text())
COHORT = np.load(SETUP / "cohort.npy")
TARGETS = np.load(SETUP / "targets.npy")
ACTIVE_NEXT = np.load(SETUP / "active_next.npy")
USER_IDS = np.load(SETUP / "user_ids.npy")
N_USERS = CONFIG["n_users"]
print("submit cohort", int(COHORT[-1].sum()), "of", N_USERS)

### Aux targets

In [ ]:
AUX_NAMES = [
    "lgmv_30",
    "lgmv_search_30",
    "lgmv_cat_30",
    "lord_30",
    "lcart_30",
    "lsearches_30",
    "n_active_30",
    "n_order_30",
    "has_order_30",
    "lgmv_7",
    "lgmv_14",
    "lgmv_60",
]
AUX_USE = [
    "lgmv_search_30",
    "lgmv_cat_30",
    "lord_30",
    "lcart_30",
    "lsearches_30",
    "n_active_30",
    "n_order_30",
    "has_order_30",
    "lgmv_7",
    "lgmv_14",
]
AUX_PATH = DATA / "aux_targets.npy"
if not AUX_PATH.exists():
    rows = np.searchsorted(USER_IDS, data["user_id"].to_numpy())
    day = (
        data
        .select((pl.col("event_date") - pl.lit(DATE_MIN)).dt.total_days().cast(pl.Int32))
        .to_series()
        .to_numpy()
    )
    src_cols = {
        c: data[c].to_numpy().astype(np.float64)
        for c in ("gmv", "gmv_search", "gmv_cat", "to_ord", "to_cart", "searches")
    }
    has_ord_row = (src_cols["to_ord"] > 0).astype(np.float64)
    aux = np.zeros((len(TRAIN_DATES), N_USERS, len(AUX_NAMES)), dtype=np.float32)
    for i, anchor in enumerate(TRAIN_DATES):
        t = (anchor - DATE_MIN).days
        m30 = (day > t) & (day <= t + 30)
        for j, name in enumerate(("gmv", "gmv_search", "gmv_cat", "to_ord", "to_cart", "searches")):
            aux[i, :, j] = np.log1p(
                np.bincount(rows[m30], weights=src_cols[name][m30], minlength=N_USERS)
            )
        aux[i, :, 6] = np.bincount(rows[m30], minlength=N_USERS)
        aux[i, :, 7] = np.bincount(rows[m30], weights=has_ord_row[m30], minlength=N_USERS)
        aux[i, :, 8] = (aux[i, :, 7] > 0).astype(np.float32)
        for j, h in ((9, 7), (10, 14), (11, 60)):
            m = (day > t) & (day <= t + h)
            aux[i, :, j] = np.log1p(
                np.bincount(rows[m], weights=src_cols["gmv"][m], minlength=N_USERS)
            )
    np.save(AUX_PATH, aux)
    (DATA / "aux_names.json").write_text(json.dumps(AUX_NAMES, indent=1))
    print("aux targets", aux.shape, "max dev", np.abs(np.log1p(TARGETS) - aux[:, :, 0]).max())
    del aux, rows, day, src_cols, has_ord_row
    gc.collect()
AUX_IDX = [AUX_NAMES.index(c) for c in AUX_USE]
print(AUX_PATH, len(AUX_IDX), "aux columns used")

### Dense grid

In [ ]:
USER_BATCH = 25_000
WINDOWS = [7, 14, 30, 60, 90, 180, 365]
HALF_LIVES = [3, 7, 14, 30, 60, 120, 240]
SUM_COLS = [
    "gmv",
    "gmv_search",
    "gmv_cat",
    "searches",
    "to_cart",
    "to_ord",
    "search_to_ord",
    "cat_to_ord",
    "cat",
    "active",
    "has_order",
    "has_cart",
    "loggmv",
]
EWM_COLS = ["gmv", "loggmv", "has_order", "active", "logsearches", "logto_cart", "logto_ord"]
EXTRA_SUM_COLS = [
    "search_only",
    "cat_only",
    "both_ch",
    "neither_ch",
    "search_to_cart",
    "cat_to_cart",
    "ord_t",
    "ord_t2",
    "loggmv2",
]
RANK_COLS = [
    "gmv_sum_30d",
    "gmv_sum_90d",
    "gmv_sum_365d",
    "to_ord_sum_30d",
    "to_ord_sum_90d",
    "to_ord_sum_365d",
    "searches_sum_30d",
    "active_sum_30d",
    "active_sum_365d",
    "has_order_sum_30d",
    "has_order_sum_365d",
    "loggmv_sum_365d",
    "days_since_last_event",
    "days_since_last_order",
    "ewm_loggmv_60",
    "ewm_loggmv_120",
    "ewm_has_order_60",
    "ewm_active_60",
    "ewm_logto_ord_60",
    "loggmv_per_order_365d",
]
V2_DENSE = [*SUM_COLS, "gmv2", "loggmv_t"]
V3_DENSE = [*V2_DENSE, *EXTRA_SUM_COLS]
CAP_COLS = [
    "days_since_last_order",
    "days_since_last_cart",
    "tenure_days",
    "days_since_first_order",
]
CAP_AT = 180.0
N_DAYS = (DATE_MAX - DATE_MIN).days + 1
ANCHOR_T = [(a - DATE_MIN).days for a in ALL_DATES]
print(N_DAYS, "days,", len(ANCHOR_T), "anchors")

In [ ]:
X_MISSING = [a for a in ALL_DATES if not (FEAT / "x" / f"X_{a.isoformat()}.npy").exists()]
E_MISSING = [a for a in ALL_DATES if not (FEAT / "e" / f"E_{a.isoformat()}.npy").exists()]
F_MISSING = [a for a in ALL_DATES if not (FEAT / "f" / f"F_{a.isoformat()}.npy").exists()]
NEED_PREP = bool(X_MISSING or E_MISSING or F_MISSING)
print(len(X_MISSING), "x,", len(E_MISSING), "e,", len(F_MISSING), "f anchors to build")

if NEED_PREP:
    UID = data["user_id"].to_numpy()
    DAY_IDX = (
        data
        .select((pl.col("event_date") - pl.lit(DATE_MIN)).dt.total_days().cast(pl.Int32))
        .to_series()
        .to_numpy()
    )
    gmv = data["gmv"].to_numpy().astype(np.float32)
    to_ord = data["to_ord"].to_numpy().astype(np.float32)
    to_cart = data["to_cart"].to_numpy().astype(np.float32)
    searches = data["searches"].to_numpy().astype(np.float32)
    cat = data["cat"].to_numpy().astype(np.float32)
    search = (searches > 0).astype(np.float32)
    has_order = (to_ord > 0).astype(np.float32)
    loggmv = np.log1p(gmv)
    fday = DAY_IDX.astype(np.float32)
    COLS = {
        "gmv": gmv,
        "gmv_search": data["gmv_search"].to_numpy().astype(np.float32),
        "gmv_cat": data["gmv_cat"].to_numpy().astype(np.float32),
        "searches": searches,
        "to_cart": to_cart,
        "to_ord": to_ord,
        "search_to_ord": data["search_to_ord"].to_numpy().astype(np.float32),
        "cat_to_ord": data["cat_to_ord"].to_numpy().astype(np.float32),
        "cat": cat,
        "active": np.ones(data.height, dtype=np.float32),
        "has_order": has_order,
        "has_cart": (to_cart > 0).astype(np.float32),
        "loggmv": loggmv,
        "gmv2": gmv * gmv,
        "logsearches": np.log1p(searches),
        "logto_cart": np.log1p(to_cart),
        "logto_ord": np.log1p(to_ord),
        "search_only": search * (1.0 - cat),
        "cat_only": (1.0 - search) * cat,
        "both_ch": search * cat,
        "neither_ch": (1.0 - search) * (1.0 - cat),
        "search_to_cart": data["search_to_cart"].to_numpy().astype(np.float32),
        "cat_to_cart": data["cat_to_cart"].to_numpy().astype(np.float32),
        "ord_t": has_order * fday,
        "ord_t2": has_order * fday * fday,
        "loggmv2": loggmv * loggmv,
        "loggmv_t": loggmv * fday,
    }
    BOUNDS = [*np.searchsorted(UID, USER_IDS[::USER_BATCH]).tolist(), len(UID)]
    del gmv, to_ord, to_cart, searches, cat, search, has_order, loggmv, fday
    gc.collect()
    print(len(BOUNDS) - 1, "user batches")

### Features

In [ ]:
N_X, N_E_BASE, N_RANK = 236, 62, 20
if X_MISSING or E_MISSING:
    XMAP, EMAP = {}, {}
    for a in ALL_DATES:
        key = a.isoformat()
        if a in X_MISSING:
            XMAP[key] = np.lib.format.open_memmap(
                FEAT / "x" / f"X_{key}.npy.part", mode="w+", dtype=np.float32, shape=(N_USERS, N_X)
            )
        if a in E_MISSING:
            EMAP[key] = np.lib.format.open_memmap(
                FEAT / "e" / f"E_{key}.npy.part",
                mode="w+",
                dtype=np.float32,
                shape=(N_USERS, N_E_BASE + N_RANK),
            )
    NAMES_X, NAMES_E = None, None
    for b in range(len(BOUNDS) - 1):
        lo, hi = BOUNDS[b], BOUNDS[b + 1]
        grow = np.searchsorted(USER_IDS, UID[lo:hi])
        base_row = int(grow[0])
        grow = grow - base_row
        nb = int(grow[-1]) + 1
        gday = DAY_IDX[lo:hi]

        CUM, KEPT = {}, {}
        for name in V3_DENSE:
            arr = np.zeros((nb, N_DAYS), dtype=np.float32)
            arr[grow, gday] = COLS[name][lo:hi]
            if name in ("active", "has_order", "has_cart") or name in EWM_COLS:
                KEPT[name] = arr
            CUM[name] = np.cumsum(arr, axis=1, dtype=np.float32)
        for name in EWM_COLS:
            if name not in KEPT:
                arr = np.zeros((nb, N_DAYS), dtype=np.float32)
                arr[grow, gday] = COLS[name][lo:hi]
                KEPT[name] = arr

        gmv_raw = KEPT["gmv"]
        run_max_gmv = np.maximum.accumulate(gmv_raw, axis=1)
        LAST = {}
        for name in ("active", "has_order", "has_cart"):
            idx = np.where(KEPT[name] > 0, np.arange(N_DAYS, dtype=np.int32), np.int32(-1))
            LAST[name] = np.maximum.accumulate(idx, axis=1)
        first_active = np.argmax(CUM["active"] > 0, axis=1).astype(np.float32)
        first_order = np.argmax(CUM["has_order"] > 0, axis=1).astype(np.float32)

        EWM = {}
        want = sorted(set(ANCHOR_T))
        for col in EWM_COLS:
            series = KEPT[col]
            for half_life in HALF_LIVES:
                alpha = 1.0 - 0.5 ** (1.0 / half_life)
                state = np.zeros(nb, dtype=np.float32)
                snaps, nxt = {}, 0
                for d in range(N_DAYS):
                    state += alpha * (series[:, d] - state)
                    while nxt < len(want) and want[nxt] == d:
                        snaps[d] = state.copy()
                        nxt += 1
                EWM[(col, half_life)] = snaps
        del KEPT
        gc.collect()

        for ai, a in enumerate(ALL_DATES):
            key = a.isoformat()
            if key not in XMAP and key not in EMAP:
                continue
            t = ANCHOR_T[ai]
            W = {}
            for col in V3_DENSE:
                for w in WINDOWS:
                    start = t - w
                    W[(col, w)] = CUM[col][:, t] - (CUM[col][:, start] if start >= 0 else 0.0)
            nx, fx = [], []
            for col in SUM_COLS:
                for w in WINDOWS:
                    nx.append(f"{col}_sum_{w}d")
                    fx.append(W[(col, w)])
            for w in (30, 90):
                n = np.maximum(W[("active", w)], 1.0)
                mean = W[("gmv", w)] / n
                nx.append(f"gmv_std_{w}d")
                fx.append(np.sqrt(np.maximum(W[("gmv2", w)] / n - mean * mean, 0.0)))
            for w in (7, 30, 90):
                nx.append(f"gmv_max_{w}d")
                fx.append(gmv_raw[:, max(t - w + 1, 0) : t + 1].max(axis=1))
            nx.append("gmv_max_life")
            fx.append(run_max_gmv[:, t])
            for w in (90, 365):
                n_span = float(min(w, t + 1))
                centre = t - (n_span - 1) / 2.0
                denom = n_span * (n_span * n_span - 1) / 12.0
                nx.append(f"loggmv_slope_{w}d")
                fx.append((W[("loggmv_t", w)] - centre * W[("loggmv", w)]) / denom)

            last_order = LAST["has_order"]
            nx.append("days_since_last_event")
            fx.append(t - LAST["active"][:, t])
            nx.append("days_since_last_order")
            fx.append(t - last_order[:, t])
            nx.append("days_since_last_cart")
            fx.append(t - LAST["has_cart"][:, t])
            nx.append("tenure_days")
            fx.append(np.where(CUM["active"][:, t] > 0, t - first_active, -1.0))
            nx.append("days_since_first_order")
            fx.append(np.where(CUM["has_order"][:, t] > 0, t - first_order, -1.0))
            nx.append("history_days")
            fx.append(np.full(nb, float(t + 1)))
            order_gap = float(min(365, t + 1)) / (W[("has_order", 365)] + 1.0)
            nx.append("order_gap_365d")
            fx.append(order_gap)
            nx.append("active_gap_90d")
            fx.append(float(min(90, t + 1)) / (W[("active", 90)] + 1.0))
            nx.append("recency_over_gap")
            fx.append((t - last_order[:, t]) / order_gap)
            last_t = last_order[:, t]
            prev_order = np.where(
                last_t >= 1, last_order[np.arange(nb), np.maximum(last_t - 1, 0)], -1
            )
            nx.append("days_since_prev_order")
            fx.append(np.where(prev_order >= 0, t - prev_order, N_DAYS))
            nx.append("last_order_gap")
            fx.append(np.where(prev_order >= 0, last_t - prev_order, N_DAYS))

            eps = 1.0
            for w in (30, 90, 365):
                nx.append(f"gmv_per_active_{w}d")
                fx.append(W[("gmv", w)] / (W[("active", w)] + eps))
                nx.append(f"basket_{w}d")
                fx.append(W[("gmv", w)] / (W[("to_ord", w)] + eps))
                nx.append(f"order_day_rate_{w}d")
                fx.append(W[("has_order", w)] / (W[("active", w)] + eps))
                nx.append(f"loggmv_per_order_{w}d")
                fx.append(W[("loggmv", w)] / (W[("has_order", w)] + eps))
                nx.append(f"loggmv_per_active_{w}d")
                fx.append(W[("loggmv", w)] / (W[("active", w)] + eps))
            for w in (7, 30, 90, 365):
                nx.append(f"active_rate_{w}d")
                fx.append(W[("active", w)] / float(min(w, t + 1)))
            for w in (30, 90):
                nx.append(f"cart_to_ord_{w}d")
                fx.append(W[("to_ord", w)] / (W[("to_cart", w)] + eps))
                nx.append(f"search_to_cart_{w}d")
                fx.append(W[("to_cart", w)] / (W[("searches", w)] + eps))
                nx.append(f"cat_gmv_share_{w}d")
                fx.append(W[("gmv_cat", w)] / (W[("gmv", w)] + eps))
                nx.append(f"cat_ord_share_{w}d")
                fx.append(W[("cat_to_ord", w)] / (W[("to_ord", w)] + eps))
            for col in ("gmv", "to_ord", "searches", "active", "loggmv"):
                nx.append(f"{col}_trend_7_30")
                fx.append(W[(col, 7)] / (W[(col, 30)] + eps))
                nx.append(f"{col}_trend_30_90")
                fx.append(W[(col, 30)] / (W[(col, 90)] + eps))
                nx.append(f"{col}_trend_90_365")
                fx.append(W[(col, 90)] / (W[(col, 365)] + eps))
                nx.append(f"{col}_m2")
                fx.append(W[(col, 60)] - W[(col, 30)])
                nx.append(f"{col}_m3")
                fx.append(W[(col, 90)] - W[(col, 60)])
            months = float(min(365, t + 1)) / 30.0
            nx.append("gmv_vs_yearly_rate")
            fx.append(W[("gmv", 30)] / (W[("gmv", 365)] / months + eps))
            nx.append("ord_vs_yearly_rate")
            fx.append(W[("to_ord", 30)] / (W[("to_ord", 365)] / months + eps))

            for col in EWM_COLS:
                for half_life in HALF_LIVES:
                    nx.append(f"ewm_{col}_{half_life}")
                    fx.append(EWM[(col, half_life)][t])
            for half_life in HALF_LIVES:
                nx.append(f"ewm_loggmv_per_active_{half_life}")
                fx.append(EWM[("loggmv", half_life)][t] / (EWM[("active", half_life)][t] + 0.01))
                nx.append(f"ewm_loggmv_per_order_{half_life}")
                fx.append(EWM[("loggmv", half_life)][t] / (EWM[("has_order", half_life)][t] + 0.01))
            for col in ("loggmv", "has_order", "active"):
                for fast, slow in ((7, 60), (14, 120), (30, 240)):
                    nx.append(f"ewm_{col}_ratio_{fast}_{slow}")
                    fx.append(EWM[(col, fast)][t] / (EWM[(col, slow)][t] + 0.01))

            block_x = np.column_stack([np.asarray(v, dtype=np.float32) for v in fx])
            assert block_x.shape == (nb, N_X), block_x.shape
            NAMES_X = nx if NAMES_X is None else NAMES_X
            assert nx == NAMES_X
            if key in XMAP:
                XMAP[key][base_row : base_row + nb] = block_x
            del block_x, fx

            if key in EMAP:
                ne, fe = [], []
                for col in (
                    "search_only",
                    "cat_only",
                    "both_ch",
                    "neither_ch",
                    "search_to_cart",
                    "cat_to_cart",
                ):
                    for w in WINDOWS:
                        ne.append(f"{col}_sum_{w}d")
                        fe.append(W[(col, w)])
                for w in (30, 90):
                    act = W[("active", w)] + eps
                    ne.append(f"search_only_share_{w}d")
                    fe.append(W[("search_only", w)] / act)
                    ne.append(f"cat_any_share_{w}d")
                    fe.append((W[("cat_only", w)] + W[("both_ch", w)]) / act)
                    ne.append(f"neither_share_{w}d")
                    fe.append(W[("neither_ch", w)] / act)
                    ne.append(f"cat_cart_share_{w}d")
                    fe.append(
                        W[("cat_to_cart", w)]
                        / (W[("search_to_cart", w)] + W[("cat_to_cart", w)] + eps)
                    )
                for w in (90, 365):
                    n = W[("has_order", w)]
                    nz = np.maximum(n, 1.0)
                    m1 = W[("ord_t", w)] / nz
                    var = np.maximum(W[("ord_t2", w)] / nz - m1 * m1, 0.0)
                    ne.append(f"order_pos_sd_{w}d")
                    fe.append(np.where(n > 1, np.sqrt(var), -1.0))
                    ne.append(f"order_centroid_recency_{w}d")
                    fe.append(np.where(n > 0, t - m1, float(N_DAYS)))
                    lg = W[("loggmv", w)]
                    lm = lg / nz
                    lvar = np.maximum(W[("loggmv2", w)] / nz - lm * lm, 0.0)
                    ne.append(f"loggmv_sd_{w}d")
                    fe.append(np.where(n > 1, np.sqrt(lvar), -1.0))
                co = CUM["has_order"]
                total = co[:, t]
                for k in (3, 5, 10):
                    target = total - (k - 1)
                    pos = np.argmax(co >= target[:, None], axis=1)
                    ne.append(f"days_since_order_{k}")
                    fe.append(np.where(total >= k, t - pos, float(N_DAYS)).astype(np.float32))
                idx = np.arange(nb)
                last_gmv = np.where(last_t >= 0, gmv_raw[idx, np.maximum(last_t, 0)], 0.0)
                prev_pos = last_order[idx, np.maximum(last_t - 1, 0)]
                prev_gmv = np.where(prev_pos >= 0, gmv_raw[idx, np.maximum(prev_pos, 0)], 0.0)
                ne.append("last_order_loggmv")
                fe.append(np.log1p(last_gmv))
                ne.append("prev_order_loggmv")
                fe.append(np.log1p(prev_gmv))
                mean_ord = W[("loggmv", 365)] / np.maximum(W[("has_order", 365)], 1.0)
                ne.append("last_over_mean_order")
                fe.append(np.log1p(last_gmv) / (mean_ord + 0.01))
                block_e = np.column_stack([np.asarray(v, dtype=np.float32) for v in fe])
                assert block_e.shape == (nb, N_E_BASE), block_e.shape
                NAMES_E = ne if NAMES_E is None else NAMES_E
                assert ne == NAMES_E
                EMAP[key][base_row : base_row + nb, :N_E_BASE] = block_e
                del block_e, fe
            del W
        del CUM, EWM, gmv_raw, run_max_gmv, LAST
        gc.collect()
        print(f"v2 v3 batch {b + 1}/{len(BOUNDS) - 1}", flush=True)

    for m in XMAP.values():
        m.flush()
    for a in X_MISSING:
        key = a.isoformat()
        del XMAP[key]
        (FEAT / "x" / f"X_{key}.npy.part").replace(FEAT / "x" / f"X_{key}.npy")
    assert NAMES_X is not None
    (FEAT / "names_x.json").write_text(json.dumps(NAMES_X))
    np.save(
        FEAT / "keep_idx.npy",
        np.array([j for j, nm in enumerate(NAMES_X) if nm != "history_days"], dtype=np.int32),
    )
    print("v2 features", len(NAMES_X))

### Rank columns

In [ ]:
if E_MISSING:
    for m in EMAP.values():
        m.flush()
    rank_src = [NAMES_X.index(c) for c in RANK_COLS]
    for a in E_MISSING:
        key = a.isoformat()
        base = np.load(FEAT / "x" / f"X_{key}.npy", mmap_mode="r")
        emap = EMAP[key]
        for j, col in enumerate(rank_src):
            v = np.asarray(base[:, col], dtype=np.float64)
            n = len(v)
            order = np.argsort(v, kind="mergesort")
            sv = v[order]
            new = np.empty(n, dtype=bool)
            new[0] = True
            np.not_equal(sv[1:], sv[:-1], out=new[1:])
            grp = np.cumsum(new) - 1
            counts = np.bincount(grp)
            starts = np.concatenate([[0], np.cumsum(counts)[:-1]])
            ranked = np.empty(n, dtype=np.float64)
            ranked[order] = starts[grp] + (counts[grp] - 1) / 2.0
            emap[:, N_E_BASE + j] = (ranked / n).astype(np.float32)
        emap.flush()
        assert np.isfinite(np.asarray(emap)).all()
        del emap
        del EMAP[key]
        (FEAT / "e" / f"E_{key}.npy.part").replace(FEAT / "e" / f"E_{key}.npy")
        print("ranks", key, flush=True)
    assert NAMES_E is not None
    NAMES_E_FULL = [*NAMES_E, *[f"rank_{c}" for c in RANK_COLS]]
    (FEAT / "names_e.json").write_text(json.dumps(NAMES_E_FULL, indent=1))
    print("v3 extra features", len(NAMES_E_FULL))

### Features

In [ ]:
N_BLOCK, BLOCK, N_WEEK, MISS, ORDER_LOOKBACK = 12, 30, 10, -1.0, 365
ORDER_NAMES = [
    "os_n_orders",
    "os_gap_mean",
    "os_gap_sd",
    "os_gap_max",
    "os_gap_min",
    "os_gap_le30",
    "os_gap_le60",
    "os_recency",
    "os_lgmv_mean",
    "os_lgmv_max",
    "os_lgmv_sd",
    "os_span",
    "os_rate",
    "os_rec_over_gap",
]


@numba.njit(parallel=True)
def order_stats(starts, ends, days, gmvs, ts, lookback, out):  # noqa: PLR0915, C901
    n_users = starts.shape[0]
    n_anchor = ts.shape[0]
    for u in numba.prange(n_users):
        lo, hi = starts[u], ends[u]
        for ai in range(n_anchor):
            t = ts[ai]
            n = 0
            s_gap = 0.0
            s_gap2 = 0.0
            g_max = 0.0
            g_min = 1e9
            le30 = 0.0
            le60 = 0.0
            s_lg = 0.0
            s_lg2 = 0.0
            lg_max = 0.0
            prev = -1
            first = -1
            last = -1
            for i in range(lo, hi):
                d = days[i]
                if d > t:
                    break
                if d <= t - lookback:
                    continue
                if first < 0:
                    first = d
                last = d
                lg = np.log1p(gmvs[i])
                s_lg += lg
                s_lg2 += lg * lg
                lg_max = max(lg_max, lg)
                if prev >= 0:
                    g = float(d - prev)
                    s_gap += g
                    s_gap2 += g * g
                    g_max = max(g_max, g)
                    g_min = min(g_min, g)
                    if g <= 30.0:
                        le30 += 1.0
                    if g <= 60.0:
                        le60 += 1.0
                prev = d
                n += 1
            o = out[ai, u]
            if n == 0:
                for j in range(14):
                    o[j] = MISS
                continue
            rec = float(t - last)
            o[0] = float(n)
            o[7] = rec
            o[8] = s_lg / n
            o[9] = lg_max
            o[10] = np.sqrt(max(s_lg2 / n - (s_lg / n) ** 2, 0.0))
            span = float(min(t - first, lookback))
            o[11] = span
            o[12] = float(n) / max(span, 1.0)
            k = n - 1
            if k <= 0:
                o[1] = MISS
                o[2] = MISS
                o[3] = MISS
                o[4] = MISS
                o[5] = MISS
                o[6] = MISS
                o[13] = MISS
                continue
            gm = s_gap / k
            o[1] = gm
            o[2] = np.sqrt(max(s_gap2 / k - gm * gm, 0.0))
            o[3] = g_max
            o[4] = g_min
            o[5] = le30 / k
            o[6] = le60 / k
            o[13] = rec / max(gm, 1.0)


NAMES_F = []
for tag in ("lgmv", "ord", "act", "has"):
    NAMES_F += [f"blk_{tag}_{k}" for k in range(N_BLOCK)]
NAMES_F += [
    "blk_lgmv_mean",
    "blk_lgmv_sd",
    "blk_lgmv_max",
    "blk_lgmv_min",
    "blk_lgmv_mean3",
    "blk_lgmv_mean6",
    "blk_lgmv_slope",
    "blk_lgmv_last_over_mean",
    "blk_has_rate",
    "blk_has_rate3",
    "blk_has_rate6",
    "blk_n_avail",
    "blk_lgmv_nz_mean",
    "blk_lgmv_gt_last",
]
for tag in ("act", "ord", "lgmv"):
    NAMES_F += [f"wk_{tag}_{w}" for w in range(N_WEEK)]
NAMES_F += [
    "streak_inactive_now",
    "streak_active_max_90",
    "streak_inactive_max_90",
    "streak_active_max_365",
]
NAMES_F += [f"dow_ord_{d}" for d in range(7)]
NAMES_F += [f"dow_act_{d}" for d in range(7)]
NAMES_F += ["dow_weekend_ord", "dow_weekend_act", "dow_ord_conc", "dow_act_conc"]
NAMES_F += ORDER_NAMES
N_F = len(NAMES_F)
print(N_F, "v4 columns")

In [ ]:
if F_MISSING:
    dow_of_day = ((np.arange(N_DAYS) + DATE_MIN.weekday()) % 7).astype(np.int64)
    FMAP = {}
    for a in F_MISSING:
        key = a.isoformat()
        FMAP[key] = np.lib.format.open_memmap(
            FEAT / "f" / f"F_{key}.npy.part", mode="w+", dtype=np.float32, shape=(N_USERS, N_F)
        )
    for b in range(len(BOUNDS) - 1):
        lo, hi = BOUNDS[b], BOUNDS[b + 1]
        grow = np.searchsorted(USER_IDS, UID[lo:hi])
        base_row = int(grow[0])
        grow = grow - base_row
        nb = int(grow[-1]) + 1
        gday = DAY_IDX[lo:hi]
        gmv = np.zeros((nb, N_DAYS), dtype=np.float32)
        gmv[grow, gday] = COLS["gmv"][lo:hi]
        to_ord = np.zeros((nb, N_DAYS), dtype=np.float32)
        to_ord[grow, gday] = COLS["to_ord"][lo:hi]
        active = np.zeros((nb, N_DAYS), dtype=np.float32)
        active[grow, gday] = COLS["active"][lo:hi]
        has_ord = (to_ord > 0).astype(np.float32)
        cum_gmv = np.cumsum(gmv, axis=1, dtype=np.float64)
        cum_ord = np.cumsum(to_ord, axis=1, dtype=np.float32)
        cum_act = np.cumsum(active, axis=1, dtype=np.float32)
        cum_has = np.cumsum(has_ord, axis=1, dtype=np.float32)

        r, c = np.nonzero(has_ord)
        ostats = np.empty((len(ANCHOR_T), nb, 14), dtype=np.float32)
        order_stats(
            np.searchsorted(r, np.arange(nb)).astype(np.int64),
            np.searchsorted(r, np.arange(nb), side="right").astype(np.int64),
            c.astype(np.int64),
            gmv[r, c].astype(np.float64),
            np.asarray(ANCHOR_T, dtype=np.int64),
            np.int64(ORDER_LOOKBACK),
            ostats,
        )

        for ai, a in enumerate(ALL_DATES):
            key = a.isoformat()
            if key not in FMAP:
                continue
            t = ANCHOR_T[ai]
            cols = []
            blk_lg, blk_or, blk_ac, blk_hs = [], [], [], []
            n_av = 0
            for k in range(N_BLOCK):
                w0, w1 = k * BLOCK, k * BLOCK + BLOCK - 1
                if t - w1 >= 0:
                    n_av += 1
                    p0 = t - w1 - 1
                    blk_lg.append(
                        np.log1p(cum_gmv[:, t - w0] - (cum_gmv[:, p0] if p0 >= 0 else 0.0)).astype(
                            np.float32
                        )
                    )
                    blk_or.append(
                        np.log1p(cum_ord[:, t - w0] - (cum_ord[:, p0] if p0 >= 0 else 0.0))
                    )
                    blk_ac.append(cum_act[:, t - w0] - (cum_act[:, p0] if p0 >= 0 else 0.0))
                    blk_hs.append(
                        ((cum_has[:, t - w0] - (cum_has[:, p0] if p0 >= 0 else 0.0)) > 0).astype(
                            np.float32
                        )
                    )
                else:
                    z = np.full(nb, MISS, dtype=np.float32)
                    blk_lg.append(z)
                    blk_or.append(z)
                    blk_ac.append(z)
                    blk_hs.append(z)
            cols += blk_lg + blk_or + blk_ac + blk_hs
            lg = np.stack(blk_lg[:n_av])
            hs = np.stack(blk_hs[:n_av])
            mblk = lg.mean(axis=0)
            xax = np.arange(n_av, dtype=np.float32)
            xc = xax - xax.mean()
            cols += [
                mblk,
                lg.std(axis=0),
                lg.max(axis=0),
                lg.min(axis=0),
                lg[: min(3, n_av)].mean(axis=0),
                lg[: min(6, n_av)].mean(axis=0),
                (lg * xc[:, None]).sum(axis=0) / max(float((xc * xc).sum()), 1.0),
                lg[0] / (mblk + 0.01),
                hs.mean(axis=0),
                hs[: min(3, n_av)].mean(axis=0),
                hs[: min(6, n_av)].mean(axis=0),
                np.full(nb, float(n_av), dtype=np.float32),
                lg.sum(axis=0) / np.maximum(hs.sum(axis=0), 1.0),
                (lg > lg[0]).sum(axis=0).astype(np.float32),
            ]
            for cum, as_log in ((cum_act, False), (cum_ord, True), (cum_gmv, True)):
                for w in range(N_WEEK):
                    w0, w1 = w * 7, w * 7 + 6
                    p0 = t - w1 - 1
                    v = cum[:, t - w0] - (cum[:, p0] if p0 >= 0 else 0.0)
                    cols.append(np.log1p(np.maximum(v, 0.0)) if as_log else v)

            inact = 1.0 - active[:, max(t - 89, 0) : t + 1]
            run = np.zeros(nb, dtype=np.float32)
            best_i = np.zeros(nb, dtype=np.float32)
            for j in range(inact.shape[1]):
                run = (run + 1.0) * inact[:, j]
                best_i = np.maximum(best_i, run)
            run_i = run
            act90 = active[:, max(t - 89, 0) : t + 1]
            run = np.zeros(nb, dtype=np.float32)
            best_a90 = np.zeros(nb, dtype=np.float32)
            for j in range(act90.shape[1]):
                run = (run + 1.0) * act90[:, j]
                best_a90 = np.maximum(best_a90, run)
            act365 = active[:, max(t - 364, 0) : t + 1]
            run = np.zeros(nb, dtype=np.float32)
            best_a365 = np.zeros(nb, dtype=np.float32)
            for j in range(act365.shape[1]):
                run = (run + 1.0) * act365[:, j]
                best_a365 = np.maximum(best_a365, run)
            cols += [run_i, best_a90, best_i, best_a365]

            d0 = max(t - 364, 0)
            dows = dow_of_day[d0 : t + 1]
            ord_w = has_ord[:, d0 : t + 1]
            act_w = active[:, d0 : t + 1]
            tot_o = np.maximum(ord_w.sum(axis=1), 1.0)
            tot_a = np.maximum(act_w.sum(axis=1), 1.0)
            dow_o = [ord_w[:, dows == d].sum(axis=1) / tot_o for d in range(7)]
            dow_a = [act_w[:, dows == d].sum(axis=1) / tot_a for d in range(7)]
            cols += dow_o + dow_a
            cols += [
                dow_o[5] + dow_o[6],
                dow_a[5] + dow_a[6],
                np.stack(dow_o).max(axis=0),
                np.stack(dow_a).max(axis=0),
            ]
            block = np.column_stack([np.asarray(v, dtype=np.float32) for v in cols])
            block = np.concatenate([block, ostats[ai]], axis=1)
            assert block.shape == (nb, N_F), block.shape
            FMAP[key][base_row : base_row + nb] = np.nan_to_num(
                block, nan=MISS, posinf=1e9, neginf=MISS
            )
            del block, cols
        del gmv, to_ord, active, has_ord, cum_gmv, cum_ord, cum_act, cum_has, ostats
        gc.collect()
        print(f"v4 batch {b + 1}/{len(BOUNDS) - 1}", flush=True)

    for m in FMAP.values():
        m.flush()
    for a in F_MISSING:
        key = a.isoformat()
        del FMAP[key]
        (FEAT / "f" / f"F_{key}.npy.part").replace(FEAT / "f" / f"F_{key}.npy")
    (FEAT / "names_f.json").write_text(json.dumps(NAMES_F, indent=1))
    print("v4 extra features", N_F)

if NEED_PREP:
    del COLS, UID, DAY_IDX
    gc.collect()

### Context

In [ ]:
NAMES_X = json.loads((FEAT / "names_x.json").read_text())
NAMES_E_FULL = json.loads((FEAT / "names_e.json").read_text())
NAMES_F = json.loads((FEAT / "names_f.json").read_text())
KEEP_IDX = np.load(FEAT / "keep_idx.npy")
BASE_NAMES = [NAMES_X[j] for j in KEEP_IDX] + NAMES_E_FULL
FEATURE_NAMES = BASE_NAMES + NAMES_F
N_BASE = len(BASE_NAMES)
CAP_IDX = np.array([BASE_NAMES.index(c) for c in CAP_COLS])
assert (N_BASE, len(FEATURE_NAMES)) == (317, 445)

MU, MU_POS = {}, {}
for a in ANCHORS:
    i = ANCHOR_IDX[a]
    m = COHORT[i]
    raw = TARGETS[i][m]
    ly = np.log1p(raw)
    MU[a] = float(ly.mean())
    MU_POS[a] = float(ly[raw > 0].mean())
print(N_BASE, len(FEATURE_NAMES), "features;", len(MU), "anchor levels")

### Rank gauss

In [ ]:
for a in [*ANCHORS, SUBMIT]:
    rg_path = FEAT / "rg" / f"RG_{a}.npy"
    if rg_path.exists():
        continue
    xa = np.asarray(np.load(FEAT / "x" / f"X_{a}.npy", mmap_mode="r"))[:, KEEP_IDX]
    ea = np.asarray(np.load(FEAT / "e" / f"E_{a}.npy", mmap_mode="r"))
    fa = np.asarray(np.load(FEAT / "f" / f"F_{a}.npy", mmap_mode="r"))
    xa = np.concatenate([xa, ea], axis=1)
    xa[:, CAP_IDX] = np.minimum(xa[:, CAP_IDX], CAP_AT)
    xa = np.concatenate([xa, fa], axis=1)
    out_rg = np.empty(xa.shape, dtype=np.float32)
    n = xa.shape[0]
    for j in range(xa.shape[1]):
        v = np.asarray(xa[:, j], dtype=np.float64)
        order = np.argsort(v, kind="mergesort")
        sv = v[order]
        new = np.empty(n, dtype=bool)
        new[0] = True
        np.not_equal(sv[1:], sv[:-1], out=new[1:])
        grp = np.cumsum(new) - 1
        counts = np.bincount(grp)
        starts = np.concatenate([[0], np.cumsum(counts)[:-1]])
        ranked = np.empty(n, dtype=np.float64)
        ranked[order] = starts[grp] + (counts[grp] - 1) / 2.0
        r = (ranked / n + 0.5 / n).clip(1e-6, 1.0 - 1e-6)
        out_rg[:, j] = ndtri(r).astype(np.float32)
    part = FEAT / "rg" / f"RG_{a}.npy.part"
    with part.open("wb") as fh:
        np.save(fh, out_rg.astype(np.float16))
    part.replace(rg_path)
    del xa, ea, fa, out_rg
    gc.collect()
    print("rank gauss", a, flush=True)
print("rank gauss cache complete")

### Daily tensors

In [ ]:
SCALE_DAYS = 90
CHANNELS9 = [
    "gmv_search",
    "gmv_cat",
    "searches",
    "search_to_cart",
    "search_to_ord",
    "cat_to_cart",
    "cat_to_ord",
    "cat",
    "active",
]
CHANNELS10 = [
    "gmv_search",
    "gmv_cat",
    "searches",
    "search_to_cart",
    "search_to_ord",
    "cat_to_cart",
    "cat_to_ord",
    "search",
    "cat",
    "active",
]
LOG_CHANNELS = {
    "gmv_search",
    "gmv_cat",
    "searches",
    "search_to_cart",
    "search_to_ord",
    "cat_to_cart",
    "cat_to_ord",
}
N_CAL = 7

for tag, channels in (("daily9", CHANNELS9), ("daily10", CHANNELS10)):
    tensor_path = SEQ / f"{tag}_f16.npy"
    if tensor_path.exists():
        continue
    uid_s = data["user_id"].to_numpy()
    day_s = (
        data
        .select((pl.col("event_date") - pl.lit(DATE_MIN)).dt.total_days().cast(pl.Int32))
        .to_series()
        .to_numpy()
    )
    chan = {}
    for name in channels:
        if name == "active":
            v = np.ones(data.height, dtype=np.float32)
        elif name == "search":
            v = (data["searches"].to_numpy().astype(np.float32) > 0).astype(np.float32)
        else:
            v = data[name].to_numpy().astype(np.float32)
        chan[name] = np.log1p(v) if name in LOG_CHANNELS else v
    scale_days = min(SCALE_DAYS, N_DAYS)
    fit = day_s < scale_days
    n_cells = N_USERS * scale_days
    scale = {}
    for name in channels:
        v = chan[name][fit]
        mean = v.sum() / n_cells
        var = (v * v).sum() / n_cells - mean * mean
        scale[name] = float(np.sqrt(max(var, 1e-12)))
    out_t = np.lib.format.open_memmap(
        tensor_path.with_suffix(".npy.part"),
        mode="w+",
        dtype=np.float16,
        shape=(N_USERS, N_DAYS, len(channels)),
    )
    seq_bounds = [*np.searchsorted(uid_s, USER_IDS[::USER_BATCH]).tolist(), len(uid_s)]
    for b in range(len(seq_bounds) - 1):
        lo, hi = seq_bounds[b], seq_bounds[b + 1]
        rows = np.searchsorted(USER_IDS, uid_s[lo:hi])
        base_row = int(rows[0])
        rows = rows - base_row
        nb = int(rows[-1]) + 1
        block = np.zeros((nb, N_DAYS, len(channels)), dtype=np.float32)
        for ci, name in enumerate(channels):
            block[rows, day_s[lo:hi], ci] = chan[name][lo:hi] / scale[name]
        out_t[base_row : base_row + nb] = block.astype(np.float16)
        del block
        print(f"{tag} batch {b + 1}/{len(seq_bounds) - 1}", flush=True)
    out_t.flush()
    del out_t
    tensor_path.with_suffix(".npy.part").replace(tensor_path)
    (SEQ / f"{tag}_meta.json").write_text(
        json.dumps(
            {
                "channels": channels,
                "scale": scale,
                "n_users": N_USERS,
                "n_days": N_DAYS,
                "scale_days": scale_days,
                "date_min": DATE_MIN.isoformat(),
            },
            indent=2,
        )
    )
    del chan, uid_s, day_s
    gc.collect()
    print(tag, "tensor written")

CAL_PATH = SEQ / "calendar.npy"
if not CAL_PATH.exists():
    di = (
        data
        .group_by("event_date")
        .agg(pl.len().alias("dau"), pl.col("gmv").sum().alias("gmv"))
        .sort("event_date")
    )
    idx = np.zeros(N_DAYS, dtype=np.float64)
    dpos = (
        di
        .select((pl.col("event_date") - pl.lit(DATE_MIN)).dt.total_days().cast(pl.Int32))
        .to_series()
        .to_numpy()
    )
    idx[dpos] = di["gmv"].to_numpy() / np.maximum(di["dau"].to_numpy(), 1)
    li = np.log1p(idx)
    li = (li - li.mean()) / max(li.std(), 1e-9)
    tt = np.arange(N_DAYS, dtype=np.float64)
    dow = (tt + DATE_MIN.weekday()) % 7
    doy = (tt + DATE_MIN.timetuple().tm_yday - 1) % 365.25
    cal = np.column_stack([
        li,
        np.sin(2 * np.pi * dow / 7),
        np.cos(2 * np.pi * dow / 7),
        np.sin(2 * np.pi * doy / 365.25),
        np.cos(2 * np.pi * doy / 365.25),
        tt / N_DAYS,
        np.ones(N_DAYS),
    ]).astype(np.float32)
    assert cal.shape == (N_DAYS, N_CAL)
    np.save(CAL_PATH, cal)
print("daily tensors and calendar ready")

data = None
gc.collect()
print("raw frame released; re-run Load if a later cell needs it")

### Members

In [ ]:
SEED_POOL = [
    42,
    7,
    2024,
    555,
    31337,
    101,
    202,
    303,
    909,
    1234,
    5678,
    4242,
    777,
    1111,
    2222,
    3333,
    8080,
    6060,
    4040,
    2020,
]
RG_KINDS = ("mlp", "mlpce", "tabm", "mlpord")
FOLDS = [*VAL_ANCHORS, SUBMIT]
FIT_ANCHORS = ["2025-09-24", "2025-10-08", "2025-10-22", "2025-12-03"]
EVAL_ANCHORS = ["2025-12-17", "2025-12-31", "2026-01-14"]
N_TRAIN_ANCHORS = 10

SEQ_ARCH = {
    "v3_seq": {
        "patch": 8,
        "n_patch": 22,
        "dim": 192,
        "layers": 6,
        "heads": 6,
        "ff": 2,
        "epochs": 8,
        "valid": False,
        "side": "v3",
        "pool13": False,
        "side_hidden": 256,
        "head_hidden": 256,
        "final_norm": True,
        "mix_anchors": True,
    },
    "seq_b": {
        "patch": 7,
        "n_patch": 52,
        "dim": 256,
        "layers": 6,
        "heads": 8,
        "ff": 3,
        "epochs": 14,
        "valid": True,
        "side": "all",
        "pool13": True,
        "side_hidden": 512,
        "head_hidden": 512,
        "final_norm": True,
        "mix_anchors": True,
    },
}
SEQ2_ARCH = {
    "patch": 7,
    "n_patch": 52,
    "dim": 256,
    "layers": 6,
    "heads": 8,
    "ff": 3,
    "side_hidden": 512,
    "head_hidden": 512,
    "final_norm": True,
    "mix_anchors": True,
}
V3_SIDE_COLS = [
    "days_since_last_event",
    "days_since_last_order",
    "days_since_last_cart",
    "tenure_days",
    "active_rate_30d",
    "active_rate_90d",
    "order_day_rate_30d",
    "loggmv_per_order_365d",
    "ewm_loggmv_60",
    "ewm_has_order_60",
    "gmv_trend_7_30",
    "gmv_trend_30_90",
]

MEMBERS = {
    "catf_a": {"kind": "cat", "v4": True, "iters": 1000, "lr": 0.03, "depth": 8, "l2": 30.0},
    "catf_b": {"kind": "cat", "v4": True, "iters": 1500, "lr": 0.03, "depth": 6, "l2": 6.0},
    "catf_d": {"kind": "cat", "v4": False, "iters": 1500, "lr": 0.03, "depth": 8, "l2": 100.0},
    "catf_f": {
        "kind": "cat",
        "v4": True,
        "iters": 1200,
        "lr": 0.03,
        "depth": 10,
        "l2": 30.0,
        "extra": {"grow_policy": "Lossguide", "max_leaves": 192, "min_data_in_leaf": 128},
    },
    "cat_a": {"kind": "cat", "v4": True, "iters": 3000, "lr": 0.03, "depth": 8},
    "cat_b": {
        "kind": "cat",
        "v4": True,
        "iters": 5000,
        "lr": 0.02,
        "depth": 10,
        "l2": 12.0,
        "border": 254,
        "sub": 0.7,
    },
    "xgbf_a": {
        "kind": "xgb",
        "v4": True,
        "rounds": 150,
        "lr": 0.03,
        "depth": 9,
        "colsample": 0.5,
        "mcw": 32.0,
        "n_oof": 3,
        "n_sub": 8,
    },
    "xgbf_b": {
        "kind": "xgb",
        "v4": False,
        "rounds": 250,
        "lr": 0.02,
        "depth": 8,
        "colsample": 0.7,
        "mcw": 100.0,
        "n_oof": 3,
        "n_sub": 8,
    },
    "xgb_a": {
        "kind": "xgb",
        "v4": True,
        "rounds": 2500,
        "lr": 0.03,
        "depth": 9,
        "colsample": 0.5,
        "mcw": 32.0,
    },
    "lgbf_a": {
        "kind": "lgb",
        "v4": False,
        "rounds": 150,
        "lr": 0.03,
        "leaves": 255,
        "n_oof": 2,
        "n_sub": 6,
    },
    "lgbf_b": {
        "kind": "lgb",
        "v4": True,
        "rounds": 250,
        "lr": 0.02,
        "leaves": 127,
        "n_oof": 2,
        "n_sub": 6,
    },
    "lgb_a": {"kind": "lgb", "v4": False, "rounds": 2500, "lr": 0.03, "leaves": 255},
    "mlpf_a": {
        "kind": "mlp",
        "v4": True,
        "epochs": 4,
        "width": 2560,
        "drop": 0.35,
        "lr": 2e-3,
        "n_oof": 6,
        "n_sub": 16,
    },
    "mlpf_b": {
        "kind": "mlp",
        "v4": False,
        "epochs": 3,
        "width": 1536,
        "drop": 0.2,
        "lr": 3e-3,
        "n_oof": 6,
        "n_sub": 16,
    },
    "mlpf_c": {"kind": "mlpce", "v4": True, "epochs": 3, "width": 1536, "n_oof": 6, "n_sub": 16},
    "mlpf_d": {
        "kind": "mlp",
        "v4": True,
        "epochs": 3,
        "width": 1024,
        "drop": 0.35,
        "lr": 3e-3,
        "n_oof": 6,
        "n_sub": 16,
    },
    "mlpf_o": {"kind": "mlpord", "v4": True, "epochs": 3, "width": 1536, "n_oof": 6, "n_sub": 16},
    "tabm_a": {
        "kind": "tabm",
        "v4": True,
        "epochs": 8,
        "width": 512,
        "blocks": 3,
        "k": 32,
        "drop": 0.1,
        "lr": 2e-3,
        "ple": False,
        "n_oof": 3,
        "n_sub": 6,
    },
    "tabm_b": {
        "kind": "tabm",
        "v4": False,
        "epochs": 4,
        "width": 512,
        "blocks": 3,
        "k": 32,
        "drop": 0.1,
        "lr": 2e-3,
        "ple": True,
        "n_oof": 2,
        "n_sub": 4,
    },
    "v3_cat": {
        "kind": "cat",
        "v4": False,
        "iters": 3000,
        "lr": 0.03,
        "depth": 8,
        "l2": 6.0,
        "n_oof": 1,
        "n_sub": 3,
    },
    "v3_cat2": {
        "kind": "cat",
        "v4": False,
        "iters": 5000,
        "lr": 0.02,
        "depth": 10,
        "l2": 12.0,
        "border": 254,
        "sub": 0.7,
        "rs": 2.0,
        "n_oof": 1,
        "n_sub": 3,
    },
    "v3_mlp": {
        "kind": "mlpv3",
        "v4": False,
        "epochs": 20,
        "width": 1024,
        "drop": 0.15,
        "lr": 3e-3,
        "n_oof": 1,
        "n_sub": 3,
    },
    "v3_seq": {"kind": "seq", "v4": False, "n_oof": 1, "n_sub": 2},
    "seq_b": {"kind": "seq", "v4": False, "n_oof": 1, "n_sub": 3},
    "seq_c": {
        "kind": "seq2",
        "v4": False,
        "epochs": 3,
        "lr": 1e-3,
        "head_lr_mult": 1.0,
        "n_oof": 1,
        "n_sub": 3,
    },
}
for m in ("catf_d", "catf_b", "xgbf_a", "catf_f", "xgbf_b", "catf_a", "lgbf_b", "lgbf_a", "cat_a"):
    MEMBERS[f"{m}_sp"] = {**MEMBERS[m], "min_gap": 28}
for cfg in MEMBERS.values():
    cfg.setdefault("min_gap", 0)
    cfg.setdefault("n_oof", 2)
    cfg.setdefault("n_sub", 3)

MEMBER_ORDER = sorted(MEMBERS)
assert len(MEMBER_ORDER) == 34
print(len(MEMBER_ORDER), "members")
print(", ".join(MEMBER_ORDER))

### Torch modules

In [ ]:
TABM_K, PLE_BINS, PLE_DIM = 32, 8, 8
HL_LO, HL_HI, HL_BINS = -2.8, 9.2, 48
HL_EDGES = np.linspace(HL_LO, HL_HI, HL_BINS + 1)
HL_CENTERS = 0.5 * (HL_EDGES[:-1] + HL_EDGES[1:])
HL_SIGMA = 0.75 * (HL_EDGES[1] - HL_EDGES[0])
ORD_LO, ORD_HI, ORD_BINS = -2.8, 9.2, 32
ORD_EDGES = np.linspace(ORD_LO, ORD_HI, ORD_BINS + 1)
ORD_CENTERS = 0.5 * (ORD_EDGES[:-1] + ORD_EDGES[1:])


class LinearBE(nn.Module):
    def __init__(self, d_in, d_out, k, first):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(d_in, d_out))
        nn.init.kaiming_uniform_(self.weight, a=5**0.5)
        sign = torch.randint(0, 2, (k, d_in), dtype=torch.float32) * 2.0 - 1.0
        self.r = nn.Parameter(sign if first else torch.ones(k, d_in))
        self.s = nn.Parameter(torch.ones(k, d_out))
        self.bias = nn.Parameter(torch.zeros(k, d_out))

    def forward(self, x):
        return ((x * self.r) @ self.weight) * self.s + self.bias


class PLE(nn.Module):
    def __init__(self, edges):
        super().__init__()
        d, n_edge = edges.shape
        t_bins = n_edge - 1
        self.register_buffer("lo", edges[:, :-1].contiguous())
        self.register_buffer("width", (edges[:, 1:] - edges[:, :-1]).clamp_min(1e-6))
        self.weight = nn.Parameter(torch.randn(d, t_bins, PLE_DIM) * (1.0 / t_bins**0.5))
        self.bias = nn.Parameter(torch.zeros(d, PLE_DIM))
        self.out_dim = d * PLE_DIM

    def forward(self, x):
        t = ((x[..., None] - self.lo) / self.width).clamp(0.0, 1.0)
        return (torch.einsum("bdt,dte->bde", t, self.weight) + self.bias).flatten(1)


class TabM(nn.Module):
    def __init__(self, d_in, k, width, blocks, drop, n_out, edges=None):
        super().__init__()
        self.k = k
        self.emb = PLE(edges) if edges is not None else None
        d = self.emb.out_dim if self.emb is not None else d_in
        self.layers = nn.ModuleList([
            LinearBE(d if i == 0 else width, width, k, i == 0) for i in range(blocks)
        ])
        self.drop = nn.Dropout(drop)
        self.head = LinearBE(width, n_out, k, False)

    def forward(self, x):
        if self.emb is not None:
            x = self.emb(x)
        h = x[:, None].expand(-1, self.k, -1)
        for lin in self.layers:
            h = self.drop(nn.functional.gelu(lin(h)))
        return self.head(h)


class Seq(nn.Module):
    def __init__(self, arch, n_side, n_ch):
        super().__init__()
        dim = arch["dim"]
        self.valid = bool(arch["valid"])
        self.pool13 = bool(arch["pool13"])
        n_extra = 2 if self.valid else 1
        n_pool = 5 if self.pool13 else 4
        self.proj = nn.Linear(2 * n_ch + n_extra, dim)
        self.pos = nn.Parameter(torch.zeros(1, arch["n_patch"], dim))
        layer = nn.TransformerEncoderLayer(
            dim,
            arch["heads"],
            dim_feedforward=arch["ff"] * dim,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.enc = nn.TransformerEncoder(
            layer, arch["layers"], norm=nn.LayerNorm(dim) if arch["final_norm"] else None
        )
        self.attn = nn.Linear(dim, 1)
        side_layers = [nn.Linear(n_side, arch["side_hidden"]), nn.GELU()]
        if self.valid:
            side_layers.append(nn.Dropout(0.15))
        side_layers.append(nn.Linear(arch["side_hidden"], dim))
        self.side = nn.Sequential(*side_layers)
        self.head = nn.Sequential(
            nn.Linear(n_pool * dim, arch["head_hidden"]),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(arch["head_hidden"], 2),
        )

    def forward(self, seq, side):
        h = self.enc(self.proj(seq) + self.pos)
        w = torch.softmax(self.attn(h), dim=1)
        pooled = [h[:, -1], h[:, -4:].mean(1)]
        if self.pool13:
            pooled.append(h[:, -13:].mean(1))
        pooled += [(h * w).sum(1), self.side(side)]
        o = self.head(torch.cat(pooled, dim=1))
        return o[:, 0], o[:, 1]


class Encoder(nn.Module):
    def __init__(self, arch, n_feat):
        super().__init__()
        dim = arch["dim"]
        self.proj = nn.Linear(n_feat, dim)
        self.pos = nn.Parameter(torch.zeros(1, arch["n_patch"], dim))
        self.mask_token = nn.Parameter(torch.zeros(1, 1, dim))
        layer = nn.TransformerEncoderLayer(
            dim,
            arch["heads"],
            dim_feedforward=arch["ff"] * dim,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.enc = nn.TransformerEncoder(
            layer, arch["layers"], norm=nn.LayerNorm(dim) if arch["final_norm"] else None
        )

    def forward(self, seq, mask=None):
        h = self.proj(seq)
        if mask is not None:
            h = torch.where(mask.unsqueeze(-1), self.mask_token.expand_as(h), h)
        return self.enc(h + self.pos)


class SeqNet(nn.Module):
    def __init__(self, arch, n_feat, n_side):
        super().__init__()
        dim = arch["dim"]
        self.encoder = Encoder(arch, n_feat)
        self.attn = nn.Linear(dim, 1)
        self.side = nn.Sequential(
            nn.Linear(n_side, arch["side_hidden"]),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(arch["side_hidden"], dim),
        )
        self.head = nn.Sequential(
            nn.Linear(5 * dim, arch["head_hidden"]),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(arch["head_hidden"], 2),
        )

    def forward(self, seq, side):
        h = self.encoder(seq)
        w = torch.softmax(self.attn(h), dim=1)
        pooled = [h[:, -1], h[:, -4:].mean(1), h[:, -13:].mean(1), (h * w).sum(1), self.side(side)]
        o = self.head(torch.cat(pooled, dim=1))
        return o[:, 0], o[:, 1]


print("torch modules defined")

### Tabular members

In [ ]:
from catboost import CatBoostRegressor, Pool
import lightgbm as lgb
import xgboost as xgb

MODEL_EXT = {
    "cat": "cbm",
    "xgb": "ubj",
    "lgb": "txt",
    "mlp": "pt",
    "mlpce": "pt",
    "mlpord": "pt",
    "mlpv3": "pt",
    "tabm": "pt",
    "seq": "pt",
    "seq2": "pt",
}
TAB_MEMBERS = [m for m in MEMBER_ORDER if MEMBERS[m]["kind"] not in ("seq", "seq2")]
for name in MEMBER_ORDER:
    (MEMBER_DIR / name).mkdir(parents=True, exist_ok=True)

PRED_DONE = {}
for name in MEMBER_ORDER:
    cfg = MEMBERS[name]
    ext = MODEL_EXT[cfg["kind"]]
    seeds_sub = SEED_POOL[: cfg["n_sub"]]
    for va in FOLDS:
        if va == SUBMIT:
            ok = (MEMBER_DIR / name / "sub.npy").exists() and all(
                (MEMBER_DIR / name / f"sub_{s}.{ext}").exists() for s in seeds_sub
            )
        else:
            ok = (MEMBER_DIR / name / f"oof_{va}.npy").exists()
        PRED_DONE[(name, va)] = ok
print(sum(PRED_DONE.values()), "of", len(PRED_DONE), "member folds already on disk")

In [ ]:
for va in FOLDS:
    is_sub = va == SUBMIT
    for min_gap in (0, 28):
        todo = [
            m for m in TAB_MEMBERS if MEMBERS[m]["min_gap"] == min_gap and not PRED_DONE[(m, va)]
        ]
        if not todo:
            continue
        if is_sub:
            eligible = list(ANCHORS)
        else:
            cut = date.fromisoformat(va) - timedelta(days=HORIZON_DAYS)
            eligible = [a for a in ANCHORS if date.fromisoformat(a) <= cut]
        if min_gap <= 0:
            tr = eligible[-N_TRAIN_ANCHORS:]
        else:
            picked = []
            for a in reversed(eligible):
                if len(picked) >= N_TRAIN_ANCHORS:
                    break
                if (
                    picked
                    and (date.fromisoformat(picked[-1]) - date.fromisoformat(a)).days < min_gap
                ):
                    continue
                picked.append(a)
            tr = sorted(picked)
        print(f"\n{va}  min_gap={min_gap}  n_train={len(tr)}  {len(todo)} members", flush=True)

        for is_rg in (False, True):
            group = [m for m in todo if (MEMBERS[m]["kind"] in RG_KINDS) == is_rg]
            if not group:
                continue
            assert va not in tr
            xs, ls, mus, mps = [], [], [], []
            for a in [*tr, va]:
                if is_rg:
                    xa = np.asarray(np.load(FEAT / "rg" / f"RG_{a}.npy", mmap_mode="r"))
                else:
                    xa = np.asarray(np.load(FEAT / "x" / f"X_{a}.npy", mmap_mode="r"))[:, KEEP_IDX]
                    xa = np.concatenate(
                        [xa, np.asarray(np.load(FEAT / "e" / f"E_{a}.npy", mmap_mode="r"))], axis=1
                    )
                    xa[:, CAP_IDX] = np.minimum(xa[:, CAP_IDX], CAP_AT)
                    xa = np.concatenate(
                        [xa, np.asarray(np.load(FEAT / "f" / f"F_{a}.npy", mmap_mode="r"))], axis=1
                    )
                if a == va:
                    xva = xa
                    break
                i = ANCHOR_IDX[a]
                m = COHORT[i]
                xs.append(xa[m])
                n = int(m.sum())
                ls.append(np.log1p(TARGETS[i][m]))
                mus.append(np.full(n, MU[a], dtype=np.float32))
                mps.append(np.full(n, MU_POS[a], dtype=np.float32))
                del xa
            xtr = np.concatenate(xs)
            ln = np.concatenate(ls).astype(np.float32)
            mu_row = np.concatenate(mus)
            mupos_row = np.concatenate(mps)
            del xs, ls, mus, mps
            gc.collect()
            y_all = (ln - mu_row).astype(np.float32)
            pos_all = (ln > 0).astype(np.float32)
            aux_pair = np.column_stack([
                pos_all,
                np.clip(ln - mu_row, -3, 6).astype(np.float32) ** 2 / 10.0,
            ]).astype(np.float32)
            print(f"  matrix {xtr.shape} {xtr.dtype}  val {xva.shape}", flush=True)

            for name in group:
                t_member = time.time()
                cfg = MEMBERS[name]
                kind = cfg["kind"]
                ext = MODEL_EXT[kind]
                seeds = SEED_POOL[: cfg["n_sub"] if is_sub else cfg["n_oof"]]
                nc = xtr.shape[1] if cfg["v4"] else N_BASE
                xt, xv = xtr[:, :nc], xva[:, :nc]
                if kind == "mlpv3":
                    rg_pair = []
                    for src in (xt, xv):
                        rg_out = np.empty(src.shape, dtype=np.float32)
                        nrow = src.shape[0]
                        for j in range(src.shape[1]):
                            v = np.asarray(src[:, j], dtype=np.float64)
                            order = np.argsort(v, kind="mergesort")
                            sv = v[order]
                            new = np.empty(nrow, dtype=bool)
                            new[0] = True
                            np.not_equal(sv[1:], sv[:-1], out=new[1:])
                            grp = np.cumsum(new) - 1
                            counts = np.bincount(grp)
                            starts = np.concatenate([[0], np.cumsum(counts)[:-1]])
                            ranked = np.empty(nrow, dtype=np.float64)
                            ranked[order] = starts[grp] + (counts[grp] - 1) / 2.0
                            r = (ranked / nrow + 0.5 / nrow).clip(1e-6, 1.0 - 1e-6)
                            rg_out[:, j] = ndtri(r).astype(np.float32)
                        rg_pair.append(rg_out)
                    xt, xv = rg_pair
                    del rg_pair
                    gc.collect()
                preds = []
                for seed in seeds:
                    t0 = time.time()
                    out_path = MEMBER_DIR / name / f"sub_{seed}.{ext}"
                    if kind == "cat":
                        params = {
                            "loss_function": "RMSE",
                            "iterations": cfg["iters"],
                            "learning_rate": cfg["lr"],
                            "task_type": "GPU",
                            "devices": "0",
                            "depth": cfg["depth"],
                            "l2_leaf_reg": cfg.get("l2", 6.0),
                            "border_count": cfg.get("border", 128),
                            "bootstrap_type": "Bernoulli",
                            "subsample": cfg.get("sub", 0.8),
                            "random_strength": cfg.get("rs", 1.0),
                            "boosting_type": "Plain",
                            "max_ctr_complexity": 0,
                            "verbose": False,
                            "allow_writing_files": False,
                            "random_seed": seed,
                        }
                        params.update(cfg.get("extra") or {})
                        booster = CatBoostRegressor(**params)
                        booster.fit(Pool(xt, y_all))
                        pred = np.asarray(booster.predict(xv), dtype=np.float64)
                        if is_sub:
                            booster.save_model(str(out_path))
                        del booster
                    elif kind == "xgb":
                        dtr = xgb.QuantileDMatrix(xt, label=y_all, max_bin=256)
                        booster = xgb.train(
                            {
                                "objective": "reg:squarederror",
                                "eval_metric": "rmse",
                                "tree_method": "hist",
                                "device": "cuda",
                                "max_depth": cfg["depth"],
                                "eta": cfg["lr"],
                                "subsample": cfg.get("subsample", 0.8),
                                "colsample_bytree": cfg.get("colsample", 0.5),
                                "colsample_bynode": 0.8,
                                "min_child_weight": cfg.get("mcw", 32.0),
                                "lambda": cfg.get("reg_lambda", 20.0),
                                "max_bin": 256,
                                "seed": seed,
                            },
                            dtr,
                            num_boost_round=cfg["rounds"],
                        )
                        pred = np.asarray(booster.inplace_predict(xv), dtype=np.float64)
                        if is_sub:
                            booster.save_model(str(out_path))
                        del booster, dtr
                    elif kind == "lgb":
                        ds = lgb.Dataset(xt, y_all, feature_name=list(FEATURE_NAMES[: xt.shape[1]]))
                        booster = lgb.train(
                            {
                                "objective": "regression",
                                "metric": "rmse",
                                "learning_rate": cfg["lr"],
                                "num_leaves": cfg["leaves"],
                                "min_data_in_leaf": cfg.get("min_data", 200),
                                "feature_fraction": cfg.get("ff", 0.4),
                                "feature_fraction_bynode": 0.7,
                                "bagging_fraction": 0.7,
                                "bagging_freq": 1,
                                "lambda_l2": cfg.get("l2", 30.0),
                                "max_bin": 127,
                                "num_threads": 10,
                                "force_col_wise": True,
                                "verbosity": -1,
                                "seed": seed,
                            },
                            ds,
                            num_boost_round=cfg["rounds"],
                        )
                        pred = np.asarray(booster.predict(xv), dtype=np.float64)
                        if is_sub:
                            booster.save_model(str(out_path))
                        del booster, ds
                    else:
                        torch.manual_seed(seed)
                        torch.cuda.manual_seed_all(seed)
                        d_in = xt.shape[1]
                        if kind == "tabm":
                            edges = None
                            if cfg["ple"]:
                                q = np.linspace(0.0, 1.0, PLE_BINS + 1)
                                sub_rows = xt[:: max(len(xt) // 200_000, 1)].astype(np.float32)
                                e = np.quantile(sub_rows, q, axis=0).T.astype(np.float32)
                                e[:, 0] -= 1e-3
                                e[:, -1] += 1e-3
                                e = np.maximum.accumulate(e, axis=1)
                                edges = torch.from_numpy(np.ascontiguousarray(e)).to("cuda")
                            model = TabM(
                                d_in, cfg["k"], cfg["width"], cfg["blocks"], cfg["drop"], 2, edges
                            ).to("cuda")
                            batch, lr_max, wd = 4096, cfg["lr"], 3e-4
                        else:
                            width = cfg["width"]
                            drop = cfg.get("drop", 0.2)
                            n_out = {
                                "mlp": 3,
                                "mlpce": HL_BINS,
                                "mlpord": ORD_BINS - 1,
                                "mlpv3": 2,
                            }[kind]
                            model = nn.ModuleDict({
                                "body": nn.Sequential(
                                    nn.Linear(d_in, width),
                                    nn.BatchNorm1d(width),
                                    nn.GELU(),
                                    nn.Dropout(drop),
                                    nn.Linear(width, width // 2),
                                    nn.BatchNorm1d(width // 2),
                                    nn.GELU(),
                                    nn.Dropout(drop),
                                    nn.Linear(width // 2, 256),
                                    nn.BatchNorm1d(256),
                                    nn.GELU(),
                                ),
                                "skip": nn.Linear(d_in, 256),
                                "head": nn.Linear(256, n_out),
                            }).to("cuda")
                            batch, lr_max, wd = 8192, cfg.get("lr", 3e-3), 1e-4
                        epochs = cfg["epochs"]
                        n = len(y_all)
                        opt = torch.optim.AdamW(model.parameters(), lr=lr_max, weight_decay=wd)
                        sched = torch.optim.lr_scheduler.OneCycleLR(
                            opt, max_lr=lr_max, total_steps=epochs * ((n + batch - 1) // batch)
                        )
                        xt_t = torch.from_numpy(xt)
                        yt_t = torch.from_numpy(y_all)
                        pt_t = torch.from_numpy(pos_all)
                        at_t = torch.from_numpy(aux_pair)
                        edges_t = torch.tensor(HL_EDGES, dtype=torch.float32, device="cuda")
                        centers_t = torch.tensor(HL_CENTERS, dtype=torch.float32, device="cuda")
                        thr_t = torch.tensor(ORD_EDGES[1:-1], dtype=torch.float32, device="cuda")
                        ord_centers = torch.tensor(ORD_CENTERS, dtype=torch.float32, device="cuda")
                        mse, bce = nn.MSELoss(), nn.BCEWithLogitsLoss()
                        for _ in range(epochs):
                            model.train()
                            perm = torch.randperm(n)
                            for kk in range(0, n, batch):
                                i = perm[kk : kk + batch]
                                xb = xt_t[i].to("cuda").float()
                                yb = yt_t[i].to("cuda")
                                if kind == "tabm":
                                    pb = pt_t[i].to("cuda")
                                    with torch.autocast("cuda", dtype=torch.bfloat16):
                                        o = model(xb).float()
                                    loss = mse(
                                        o[:, :, 0], yb[:, None].expand(-1, cfg["k"])
                                    ) + 0.3 * bce(o[:, :, 1], pb[:, None].expand(-1, cfg["k"]))
                                elif kind == "mlpce":
                                    cdf = torch.special.ndtr(
                                        (edges_t[None, :] - yb[:, None]) / HL_SIGMA
                                    )
                                    tgt = cdf[:, 1:] - cdf[:, :-1]
                                    tgt = tgt / tgt.sum(dim=1, keepdim=True).clamp_min(1e-8)
                                    with torch.autocast("cuda", dtype=torch.bfloat16):
                                        h = model["body"](xb) + model["skip"](xb)
                                        logits = model["head"](h).float()
                                    loss = (
                                        -(tgt * torch.log_softmax(logits, dim=1)).sum(dim=1).mean()
                                    )
                                elif kind == "mlpord":
                                    with torch.autocast("cuda", dtype=torch.bfloat16):
                                        h = model["body"](xb) + model["skip"](xb)
                                        logits = model["head"](h).float()
                                    tgt = (yb[:, None] > thr_t).float()
                                    act = torch.cat(
                                        [torch.ones_like(tgt[:, :1]), tgt[:, :-1]], dim=1
                                    )
                                    raw_l = nn.functional.binary_cross_entropy_with_logits(
                                        logits, tgt, reduction="none"
                                    )
                                    loss = (raw_l * act).sum() / act.sum().clamp_min(1.0)
                                elif kind == "mlpv3":
                                    pb = pt_t[i].to("cuda")
                                    with torch.autocast("cuda", dtype=torch.bfloat16):
                                        h = model["body"](xb) + model["skip"](xb)
                                        o = model["head"](h)
                                        loss = mse(o[:, 0].float(), yb) + 0.3 * bce(
                                            o[:, 1].float(), pb
                                        )
                                else:
                                    ab = at_t[i].to("cuda")
                                    with torch.autocast("cuda", dtype=torch.bfloat16):
                                        h = model["body"](xb) + model["skip"](xb)
                                        o = model["head"](h)
                                        loss = mse(o[:, 0].float(), yb) + 0.3 * mse(
                                            o[:, 1:].float(), ab
                                        )
                                opt.zero_grad(set_to_none=True)
                                loss.backward()
                                if kind == "tabm":
                                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                                opt.step()
                                sched.step()
                        model.eval()
                        chunk = 8192 if kind == "tabm" else 16384
                        out_chunks = []
                        with torch.no_grad():
                            for kk in range(0, len(xv), chunk):
                                xb = torch.from_numpy(xv[kk : kk + chunk]).to("cuda").float()
                                with torch.autocast("cuda", dtype=torch.bfloat16):
                                    if kind == "tabm":
                                        o = model(xb).float()
                                    else:
                                        h = model["body"](xb) + model["skip"](xb)
                                        o = model["head"](h).float()
                                if kind == "tabm":
                                    out_chunks.append(o[:, :, 0].mean(dim=1).cpu().numpy())
                                elif kind == "mlpce":
                                    out_chunks.append(
                                        (torch.softmax(o, dim=1) @ centers_t).cpu().numpy()
                                    )
                                elif kind == "mlpord":
                                    p = torch.sigmoid(o)
                                    surv = torch.cat(
                                        [torch.ones_like(p[:, :1]), torch.cumprod(p, dim=1)], dim=1
                                    )
                                    nxt = torch.cat(
                                        [torch.cumprod(p, dim=1), torch.zeros_like(p[:, :1])], dim=1
                                    )
                                    prob = surv - nxt
                                    out_chunks.append(
                                        (
                                            prob
                                            / prob.sum(dim=1, keepdim=True).clamp_min(1e-8)
                                            @ ord_centers
                                        )
                                        .cpu()
                                        .numpy()
                                    )
                                else:
                                    out_chunks.append(o[:, 0].cpu().numpy())
                        pred = np.concatenate(out_chunks).astype(np.float64)
                        if is_sub:
                            torch.save(model.state_dict(), out_path)
                        del model, xt_t, yt_t, pt_t, at_t
                        torch.cuda.empty_cache()
                    preds.append(pred)
                    print(f"    {name} seed {seed} {time.time() - t0:.0f}s", flush=True)
                    gc.collect()
                raw = np.mean(preds, axis=0)
                out_np = (
                    (MEMBER_DIR / name / "sub.npy")
                    if is_sub
                    else (MEMBER_DIR / name / f"oof_{va}.npy")
                )
                np.save(out_np, raw.astype(np.float32))
                PRED_DONE[(name, va)] = True
                print(f"  {name:<12}{time.time() - t_member:>7.0f}s", flush=True)
                del preds, raw, xt, xv
                gc.collect()
            del xtr, xva, ln, mu_row, mupos_row, y_all, pos_all, aux_pair
            gc.collect()
print("tabular members done")

### Sequence members

In [ ]:
SEQ_MEMBERS = ["v3_seq", "seq_b", "seq_c"]
SIDE_IDX = {}
SIDE_IDX["all"] = np.arange(N_BASE)
SIDE_IDX["v3"] = np.array([
    BASE_NAMES.index(c) for c in [n for n in BASE_NAMES if n.startswith("rank_")] + V3_SIDE_COLS
])

for name in SEQ_MEMBERS:
    cfg = MEMBERS[name]
    if all(PRED_DONE[(name, va)] for va in FOLDS):
        print(f"{name} already complete")
        continue
    is_seq2 = cfg["kind"] == "seq2"
    arch = SEQ2_ARCH if is_seq2 else SEQ_ARCH[name]
    tensor_tag = "daily10" if is_seq2 else "daily9"
    meta = json.loads((SEQ / f"{tensor_tag}_meta.json").read_text())
    n_ch = len(meta["channels"])
    side_idx = SIDE_IDX["all"] if is_seq2 else SIDE_IDX[str(arch["side"])]
    n_side = len(side_idx)
    n_content = 2 * n_ch + 1
    n_feat = n_content + 1 + N_CAL
    patch = int(arch["patch"])
    n_patch = int(arch["n_patch"])
    context_len = patch * n_patch
    daily = torch.from_numpy(np.load(SEQ / f"{tensor_tag}_f16.npy")).to("cuda")
    cal = torch.from_numpy(np.load(CAL_PATH)).to("cuda")
    print(f"{name}: {tensor_tag} on gpu {tuple(daily.shape)} side {n_side}", flush=True)

    for va in FOLDS:
        if PRED_DONE[(name, va)]:
            continue
        is_sub = va == SUBMIT
        if is_sub:
            eligible = list(ANCHORS)
        else:
            cut = date.fromisoformat(va) - timedelta(days=HORIZON_DAYS)
            eligible = [a for a in ANCHORS if date.fromisoformat(a) <= cut]
        tr = eligible[-N_TRAIN_ANCHORS:]
        train_sets = []
        for a in [*tr, va]:
            side_np = np.asarray(np.load(FEAT / "rg" / f"RG_{a}.npy", mmap_mode="r"))[:, side_idx]
            t_a = ANCHOR_T[[d.isoformat() for d in ALL_DATES].index(a)]
            if a == va:
                val_set = (
                    torch.arange(side_np.shape[0], device="cuda"),
                    t_a,
                    None,
                    None,
                    torch.from_numpy(np.ascontiguousarray(side_np)).to("cuda"),
                )
                break
            i = ANCHOR_IDX[a]
            m = COHORT[i]
            ly = np.log1p(TARGETS[i][m])
            train_sets.append((
                torch.from_numpy(np.where(m)[0]).to("cuda"),
                t_a,
                torch.from_numpy((ly - MU[a]).astype(np.float32)).to("cuda"),
                torch.from_numpy((TARGETS[i][m] > 0).astype(np.float32)).to("cuda"),
                torch.from_numpy(np.ascontiguousarray(side_np[m])).to("cuda"),
            ))
            del side_np
        gc.collect()

        preds = []
        for seed in SEED_POOL[: cfg["n_sub"] if is_sub else cfg["n_oof"]]:
            t0 = time.time()
            torch.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
            if is_seq2:
                net = SeqNet(arch, n_feat, n_side).to("cuda")
                enc_p = list(net.encoder.parameters())
                other_p = [p for nm, p in net.named_parameters() if not nm.startswith("encoder.")]
                base_lr = cfg["lr"]
                head_lr = base_lr * cfg["head_lr_mult"]
                opt = torch.optim.AdamW(
                    [{"params": enc_p, "lr": base_lr}, {"params": other_p, "lr": head_lr}],
                    weight_decay=0.01,
                )
                epochs = int(cfg["epochs"])
                steps = epochs * sum((len(s[0]) + 2047) // 2048 for s in train_sets)
                sched = torch.optim.lr_scheduler.OneCycleLR(
                    opt, max_lr=[base_lr, head_lr], total_steps=steps
                )
            else:
                net = Seq(arch, n_side, n_ch).to("cuda")
                opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=0.01)
                epochs = int(arch["epochs"])
                steps = epochs * sum((len(s[0]) + 2047) // 2048 for s in train_sets)
                sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=steps)
            mse, bce = nn.MSELoss(), nn.BCEWithLogitsLoss()
            for _ in range(epochs):
                net.train()
                order = []
                for si, s in enumerate(train_sets):
                    perm = torch.randperm(len(s[0]), device="cuda")
                    order += [(si, perm[k : k + 2048]) for k in range(0, len(perm), 2048)]
                if bool(arch["mix_anchors"]):
                    order = [order[j] for j in torch.randperm(len(order)).tolist()]
                for si, i in order:
                    users, t_a, y_s, pos_s, side_s = train_sets[si]
                    u = users[i]
                    lo_d = t_a - context_len + 1
                    if lo_d >= 0:
                        win = daily[u, lo_d : t_a + 1].float()
                        valid = torch.ones(len(u), n_patch, 1, device="cuda")
                        cwin = cal[lo_d : t_a + 1]
                    else:
                        pad = -lo_d
                        win = torch.zeros(len(u), context_len, n_ch, device="cuda")
                        win[:, pad:] = daily[u, 0 : t_a + 1].float()
                        v = torch.ones(context_len, device="cuda")
                        v[:pad] = 0.0
                        valid = (
                            v.view(n_patch, patch).amax(dim=1).view(1, -1, 1).expand(len(u), -1, -1)
                        )
                        if is_seq2:
                            cwin = torch.zeros(context_len, cal.shape[1], device="cuda")
                            cwin[pad:] = cal[0 : t_a + 1]
                    win = win.view(len(u), n_patch, patch, n_ch)
                    if is_seq2:
                        cnt = (win[:, :, :, -1] > 0).float().sum(dim=2, keepdim=True) / patch
                        content = torch.cat([win.sum(dim=2), win.amax(dim=2), cnt], dim=2)
                        cpatch = (
                            cwin
                            .view(n_patch, patch, -1)
                            .mean(dim=1)
                            .unsqueeze(0)
                            .expand(len(u), -1, -1)
                        )
                        seq = torch.cat([content, valid, cpatch], dim=2)
                    else:
                        cnt = (win[:, :, :, -1] > 0).float().sum(dim=2, keepdim=True)
                        parts = [win.sum(dim=2), win.amax(dim=2), cnt]
                        if bool(arch["valid"]):
                            parts.append(valid)
                        seq = torch.cat(parts, dim=2)
                    with torch.autocast("cuda", dtype=torch.bfloat16):
                        mu_out, logit = net(seq, side_s[i].float())
                        loss = mse(mu_out.float(), y_s[i]) + 0.3 * bce(logit.float(), pos_s[i])
                    opt.zero_grad(set_to_none=True)
                    loss.backward()
                    nn.utils.clip_grad_norm_(net.parameters(), 1.0)
                    opt.step()
                    sched.step()
            net.eval()
            vu, vt, _, _, vside = val_set
            out_chunks = []
            with torch.no_grad():
                for k in range(0, len(vu), 4096):
                    u = vu[k : k + 4096]
                    lo_d = vt - context_len + 1
                    if lo_d >= 0:
                        win = daily[u, lo_d : vt + 1].float()
                        valid = torch.ones(len(u), n_patch, 1, device="cuda")
                        cwin = cal[lo_d : vt + 1]
                    else:
                        pad = -lo_d
                        win = torch.zeros(len(u), context_len, n_ch, device="cuda")
                        win[:, pad:] = daily[u, 0 : vt + 1].float()
                        v = torch.ones(context_len, device="cuda")
                        v[:pad] = 0.0
                        valid = (
                            v.view(n_patch, patch).amax(dim=1).view(1, -1, 1).expand(len(u), -1, -1)
                        )
                        if is_seq2:
                            cwin = torch.zeros(context_len, cal.shape[1], device="cuda")
                            cwin[pad:] = cal[0 : vt + 1]
                    win = win.view(len(u), n_patch, patch, n_ch)
                    if is_seq2:
                        cnt = (win[:, :, :, -1] > 0).float().sum(dim=2, keepdim=True) / patch
                        content = torch.cat([win.sum(dim=2), win.amax(dim=2), cnt], dim=2)
                        cpatch = (
                            cwin
                            .view(n_patch, patch, -1)
                            .mean(dim=1)
                            .unsqueeze(0)
                            .expand(len(u), -1, -1)
                        )
                        seq = torch.cat([content, valid, cpatch], dim=2)
                    else:
                        cnt = (win[:, :, :, -1] > 0).float().sum(dim=2, keepdim=True)
                        parts = [win.sum(dim=2), win.amax(dim=2), cnt]
                        if bool(arch["valid"]):
                            parts.append(valid)
                        seq = torch.cat(parts, dim=2)
                    with torch.autocast("cuda", dtype=torch.bfloat16):
                        mu_out, _ = net(seq, vside[k : k + 4096].float())
                    out_chunks.append(mu_out.float().cpu().numpy())
            preds.append(np.concatenate(out_chunks).astype(np.float64))
            if is_sub:
                torch.save(net.state_dict(), MEMBER_DIR / name / f"sub_{seed}.pt")
            del net
            torch.cuda.empty_cache()
            print(f"    {name} {va} seed {seed} {time.time() - t0:.0f}s", flush=True)
        raw = np.mean(preds, axis=0)
        out_np = (
            (MEMBER_DIR / name / "sub.npy") if is_sub else (MEMBER_DIR / name / f"oof_{va}.npy")
        )
        np.save(out_np, raw.astype(np.float32))
        PRED_DONE[(name, va)] = True
        del train_sets, val_set, preds, raw
        torch.cuda.empty_cache()
        gc.collect()
    del daily, cal
    torch.cuda.empty_cache()
    gc.collect()
print("sequence members done")

### Panel

In [ ]:
PANEL_ANCHORS = [*FIT_ANCHORS, *EVAL_ANCHORS, SUBMIT]
CACHE_M, CACHE_Y = {}, {}
for a in PANEL_ANCHORS:
    m = COHORT[-1] if a == SUBMIT else COHORT[ANCHOR_IDX[a]]
    cols = []
    for name in MEMBER_ORDER:
        f = (MEMBER_DIR / name / "sub.npy") if a == SUBMIT else (MEMBER_DIR / name / f"oof_{a}.npy")
        raw = np.load(f).astype(np.float64)[m]
        sd = raw.std()
        cols.append((raw - raw.mean()) / (sd if sd > 1e-12 else 1.0))
    CACHE_M[a] = np.column_stack(cols)
    if a != SUBMIT:
        y = np.log1p(TARGETS[ANCHOR_IDX[a]][m])
        CACHE_Y[a] = (y - y.mean()) / y.std()
    print(f"{a} {CACHE_M[a].shape}", flush=True)

Xf = np.vstack([CACHE_M[a] for a in FIT_ANCHORS])
yf = np.concatenate([CACHE_Y[a] for a in FIT_ANCHORS])
print(Xf.shape, "fitting rows")

### Linear blend

In [ ]:
w_nnls_all, _ = nnls(Xf, yf)
KEEP = np.flatnonzero(w_nnls_all > 1e-8)
KEPT_MEMBERS = [MEMBER_ORDER[i] for i in KEEP]
SHIPPED_KEPT = [
    "cat_a_sp",
    "cat_b",
    "catf_b",
    "catf_b_sp",
    "catf_f",
    "catf_f_sp",
    "seq_b",
    "seq_c",
    "v3_mlp",
    "v3_seq",
    "xgb_a",
    "xgbf_a_sp",
]
SCOPES = {"kept": KEEP, "all": np.arange(len(MEMBER_ORDER))}
print(f"non-negative solve keeps {len(KEEP)} of {len(MEMBER_ORDER)}")
print("  kept:   " + ", ".join(KEPT_MEMBERS))
print("  shipped:" + ", ".join(SHIPPED_KEPT))
print("  match:", KEPT_MEMBERS == SHIPPED_KEPT)

### Residual combiners

In [ ]:
TUNED_FAMILY = {"kept": "lgbm", "all": "xgb"}
TUNED_FILE = {"kept": "v46_lgbm_residual.txt", "all": "v47_xgb_residual.ubj"}
TUNED_ROUNDS = {"kept": 151, "all": 230}
TUNED_PARAMS = {
    "kept": {
        "learning_rate": 0.010416117992371583,
        "num_leaves": 7,
        "min_data_in_leaf": 509,
        "feature_fraction": 0.6326418272631035,
        "bagging_fraction": 0.5689126760755702,
        "lambda_l2": 4.460350509886045,
    },
    "all": {
        "eta": 0.01033334897515964,
        "max_depth": 2,
        "min_child_weight": 42.975658562793505,
        "subsample": 0.7134313151821792,
        "colsample_bytree": 0.6845696095993247,
        "lambda": 0.2142552553768538,
    },
}
COMBINER_SEED = 42
BOOSTERS, NNLS_W = {}, {}
for scope, cols in SCOPES.items():
    Xc = Xf[:, cols]
    w_scope, _ = nnls(Xc, yf)
    lin = Xc @ w_scope
    NNLS_W[scope] = w_scope
    (COMBINER_DIR / f"nnls_{scope}.json").write_text(
        json.dumps(
            {
                "scope": scope,
                "members": [MEMBER_ORDER[i] for i in cols],
                "columns": [int(i) for i in cols],
                "weights": [float(v) for v in w_scope],
            },
            indent=1,
        )
    )
    xs = np.column_stack([Xc, lin])
    ys = yf - lin
    booster_path = COMBINER_DIR / TUNED_FILE[scope]
    if TUNED_FAMILY[scope] == "lgbm" and booster_path.exists():
        booster = lgb.Booster(model_file=str(booster_path))
    elif TUNED_FAMILY[scope] == "lgbm":
        booster = lgb.train(
            {
                "objective": "regression",
                "verbose": -1,
                "seed": COMBINER_SEED,
                "num_threads": 16,
                "force_row_wise": True,
                "bagging_freq": 1,
                **TUNED_PARAMS[scope],
            },
            lgb.Dataset(xs, label=ys),
            num_boost_round=TUNED_ROUNDS[scope],
        )
        booster.save_model(str(booster_path))
    elif booster_path.exists():
        booster = xgb.Booster()
        booster.load_model(str(booster_path))
    else:
        booster = xgb.train(
            {
                "objective": "reg:squarederror",
                "tree_method": "hist",
                "device": "cuda",
                "seed": COMBINER_SEED,
                **TUNED_PARAMS[scope],
            },
            xgb.DMatrix(xs, label=ys),
            num_boost_round=TUNED_ROUNDS[scope],
        )
        booster.save_model(str(booster_path))
    BOOSTERS[scope] = booster
    print(
        f"{scope:>5}  {TUNED_FAMILY[scope]} on residual, {len(cols)} members -> {booster_path.name}"
    )

(COMBINER_DIR / "members.json").write_text(
    json.dumps(
        {
            "members": MEMBER_ORDER,
            "kept": KEPT_MEMBERS,
            "family": TUNED_FAMILY,
            "file": TUNED_FILE,
            "rounds": TUNED_ROUNDS,
            "params": TUNED_PARAMS,
        },
        indent=1,
    )
)

### Forward check

In [ ]:
RECORDED = {"linear": 0.6780457242978827, "kept": 0.6778257186532514, "all": 0.6778140406740206}
FORWARD = {}
pz, py = [], []
for a in EVAL_ANCHORS:
    p = CACHE_M[a] @ w_nnls_all
    pz.append((p - p.mean()) / p.std())
    py.append(CACHE_Y[a])
FORWARD["linear"] = float(np.corrcoef(np.concatenate(pz), np.concatenate(py))[0, 1])
for scope, cols in SCOPES.items():
    pz = []
    for a in EVAL_ANCHORS:
        Z = CACHE_M[a][:, cols]
        lz = Z @ NNLS_W[scope]
        xs = np.column_stack([Z, lz])
        f = BOOSTERS[scope].predict(xs if scope == "kept" else xgb.DMatrix(xs))
        p = lz + f
        pz.append((p - p.mean()) / p.std())
    FORWARD[scope] = float(np.corrcoef(np.concatenate(pz), np.concatenate(py))[0, 1])

print(f"{'combination':<22}{'pooled rho':>12}{'recorded':>12}{'delta':>11}")
for tag, label in (
    ("linear", "linear nnls"),
    ("kept", "lgbm on residual"),
    ("all", "xgb on residual"),
):
    print(
        f"{label:<22}{FORWARD[tag]:>12.7f}{RECORDED[tag]:>12.7f}{FORWARD[tag] - RECORDED[tag]:>+11.7f}"
    )

### Member scores

In [ ]:
MEMBER_CV = {}
for name in MEMBER_ORDER:
    num, den = 0.0, 0
    per = {}
    for a in VAL_ANCHORS:
        m = COHORT[ANCHOR_IDX[a]]
        y = np.log1p(TARGETS[ANCHOR_IDX[a]][m])
        r = np.load(MEMBER_DIR / name / f"oof_{a}.npy").astype(np.float64)[m]
        pr = np.clip(r - r.mean() + y.mean(), 0.0, None)
        per[a] = float(np.sqrt(((pr - y) ** 2).mean()))
        num += float(((pr - y) ** 2).sum())
        den += len(y)
    MEMBER_CV[name] = {**per, "POOLED_ALL": float(np.sqrt(num / den))}
    (MEMBER_DIR / name / "cv.json").write_text(json.dumps(MEMBER_CV[name], indent=1))

print(f"{'member':<14}{'pooled':>10}")
for name in sorted(MEMBER_ORDER, key=lambda n: MEMBER_CV[n]["POOLED_ALL"]):
    print(f"{name:<14}{MEMBER_CV[name]['POOLED_ALL']:>10.5f}")

### Log

In [ ]:
ARCHITECTURE = (
    "34 member ensemble over the 445 column cohort by anchor space: 13 CatBoost, 5 LightGBM, "
    "5 XGBoost, 4 multi-task MLP, 1 HL-Gauss cross-entropy MLP, 1 ordinal MLP, 2 TabM batch "
    "ensembles and 3 patch transformers over the daily tensor, nine of the tree members trained "
    "a second time on spaced anchors. A non-negative least squares solve over four fit anchors "
    "gives the linear blend and the twelve member kept scope; LightGBM on the residual of the "
    "kept scope gives v46 and XGBoost on the residual of all 34 gives v47. The submission is "
    "v43 plus the parts of v46 and v47 that lie outside the span of the 63 scored predecessors, "
    "each shrunk by its own significance."
)
with mlflow.start_run(run_name="prod-train"):
    mlflow.set_tags({"pipeline": "prod", "stage": "train", "target": "submit_v48_newdir"})
    mlflow.log_params({
        "members": ",".join(MEMBER_ORDER),
        "n_members": len(MEMBER_ORDER),
        "kept_members": ",".join(KEPT_MEMBERS),
        "fit_anchors": ",".join(FIT_ANCHORS),
        "eval_anchors": ",".join(EVAL_ANCHORS),
        "submit_anchor": SUBMIT,
        "n_train_anchors": N_TRAIN_ANCHORS,
        "n_features": len(FEATURE_NAMES),
        "n_users": N_USERS,
        "combiner_kept": "lgbm on residual",
        "combiner_all": "xgb on residual",
        "architecture": ARCHITECTURE,
    })
    mlflow.log_metrics({f"member_{k}": v["POOLED_ALL"] for k, v in MEMBER_CV.items()})
    mlflow.log_metrics({f"forward_rho_{k}": v for k, v in FORWARD.items()})
    mlflow.log_metrics({f"forward_rho_recorded_{k}": v for k, v in RECORDED.items()})
    mlflow.log_metric("public_lb_rmsle", 1.6460816563)
    for scope in SCOPES:
        mlflow.log_artifact(str(COMBINER_DIR / f"nnls_{scope}.json"), "combiner")
    mlflow.log_artifact(str(COMBINER_DIR / "members.json"), "combiner")
    mlflow.log_dict(MEMBER_CV, "member_cv.json")
print("logged")